In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pickle
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

In [ ]:
stepwise_results = list(Path("outputs/trf_stepwise").glob("*/results.pkl"))
outdir = "."

In [ ]:
stepwise_results = [Path(p) for p in stepwise_results]
outdir = Path(outdir)

In [ ]:
all_results = []
for path in tqdm(stepwise_results):
    subject = path.parent.name
    with path.open("rb") as f:
        st = pickle.load(f)

    for progression in st:
        if st[progression]["ftest_results"] is None:
            # no matching electrodes here, so we skipped the analysis. all good
            continue

        for ftest_results in st[progression]["ftest_results"]:
            # get the more advanced model in this comparison ("step2")
            cv_results = st[progression]["cv_results"][ftest_results["step2"]]
            fit_electrodes = cv_results["cv_results"]["estimator"][0].estimator.fit_electrodes

            for i, elec in enumerate(fit_electrodes):
                all_results.append({
                    "subject": subject,
                    "progression": progression,
                    "progression_step": cv_results["progression_step"],
                    "electrode": elec,
                    "r2_pre": st[progression]["cv_results"][ftest_results["step1"]]["r2"][i],
                    "r2_post": cv_results["r2"][i],
                    "rss_pre": ftest_results["rss_1"][i],
                    "rss_post": ftest_results["rss_2"][i],
                    "df_diff": ftest_results["df_diff"],
                    "df_residual": ftest_results["df_residual"],
                    "ftest_stat": ftest_results["f_stat"][i],
                    "ftest_pval": ftest_results["p_value"][i],
                })

In [ ]:
all_results_df = pd.DataFrame(all_results)
all_results_df["rss_diff"] = all_results_df["rss_post"] - all_results_df["rss_pre"]

r2_valid_df = all_results_df.loc[all_results_df.r2_pre > 0]
all_results_df.loc[all_results_df.r2_pre > 0, "r2_diff"] = r2_valid_df["r2_post"] - r2_valid_df["r2_pre"]
all_results_df.sort_values("ftest_pval")

In [ ]:
g = sns.displot(data=all_results_df.assign(progression_step=all_results_df.progression_step.astype(str)),
                x="rss_diff",
                hue="progression_step", row="progression",
                height=2, aspect=3, kind="kde", fill=True)
for ax in g.axes.flat:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)

In [ ]:
g = sns.displot(data=all_results_df.assign(progression_step=all_results_df.progression_step.astype(str)),
                x="r2_diff",
                hue="progression_step", row="progression",
                height=2, aspect=3, kind="kde", fill=True)
for ax in g.axes.flat:
    ax.axvline(0, color="black", linestyle="--", linewidth=1)

In [ ]:
all_results_df.loc[all_results_df.ftest_pval < 0.05].sort_values("ftest_pval")

In [ ]:
all_results_df.loc[all_results_df.r2_diff > 0].sort_values("r2_diff", ascending=False)

In [ ]:
all_results_df.to_csv(outdir / "eois.csv", index=False)